## Import Libraries

In [1]:
import __future__
import keras

from __future__ import print_function
from keras.utils import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding
from keras.layers import SimpleRNN
from keras.datasets import imdb
from keras_preprocessing.text import Tokenizer
from keras import initializers

## Utility

In [2]:
def decode_review(encoded_review):
    """การทำงานของโค๊ดนี้คือ การนำ key (ที่ผ่านการสลับ key - value มาแล้ว)ที่เป็น unique id ของคำใน dict มา map กลับเป็นคำ"""
    # Bad index ให้ใช้คำว่า "?" แทน
    decoded_review = " ".join([inv_word_index.get(i, "?") for i in encoded_review])
    return decoded_review

## Load imdb Data

In [3]:
max_features = 20000
maxlen = 80
batch_size = 32

In [4]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

In [5]:
print(len(X_train), 'train sequences')
print(len(X_test), 'test sequences')

25000 train sequences
25000 test sequences


## Decode data into Sentence

In [6]:
# get_word_index เป็นการสร้าง dictionary ในการ map คำ กับ unique integer
word_index = imdb.get_word_index()
word_index["<PAD>"] = -3 # Padding คำให้ length เท่ากัน เพราะเรากำหนด max len เอาไว้
word_index["<START>"] = -2 # ระบุว่าเป็นคำเริ่มต้นของประโยค
word_index["<UNK>"] = -1 # คำที่ไม่รู้จัก หมายถึงเป็นคำที่ยังไม่ได้อยู่ใน Vocabulary
word_index["<UNUSED>"] = 0 # คำที่ไม่ได้ใช้
# + 3 เกิดจาก word_index ใช้ 3 ค่าแรกไปแล้วในการนิยามคำ ต่อจากนั้นถ้าเริ่มนับคำ ก็จะมีการ + 3 เข้าไปเพื่อไม่ให้ค่าทับกฎที่วางเอาไว้
# [ขยาย] word_index ที่มีการ map dictionary เอาไว้นั้นจะเริ่มที่ 1 ก่อน เราเลย + 3 เข้าไปด้วย

# Original word_index: {"the": 1, "and": 2}
# inv_word_index becomes: {4: "the", 5: "and"}

inv_word_index = {v + 3: k for k, v in word_index.items()}

In [10]:
word_index['<START>']

-2

In [11]:
inv_word_index[4]

'the'

In [14]:
for key, value in inv_word_index.items():
    print(f"{key}: {value}")

34704: fawn
52009: tsukino
52010: nunnery
16819: sonja
63954: vani
1411: woods
16118: spiders
2348: hanging
2292: woody
52011: trawling
52012: hold's
11310: comically
40833: localized
30571: disobeying
52013: 'royale
40834: harpo's
52014: canet
19316: aileen
52015: acurately
52016: diplomat's
25245: rickman
6749: arranged
52017: rumbustious
52018: familiarness
52019: spider'
68807: hahahah
52020: wood'
40836: transvestism
34705: hangin'
2341: bringing
40837: seamier
34706: wooded
52021: bravora
16820: grueling
1639: wooden
16821: wednesday
52022: 'prix
34707: altagracia
52023: circuitry
11588: crotch
57769: busybody
52024: tart'n'tangy
14132: burgade
52026: thrace
11041: tom's
52028: snuggles
29117: francesco
52030: complainers
52128: templarios
40838: 272
52031: 273
52133: zaniacs
34709: 275
27634: consenting
40839: snuggled
15495: inanimate
52033: uality
11929: bronte
4013: errors
3233: dialogs
52034: yomada's
34710: madman's
30588: dialoge
52036: usenet
40840: videodrome
26341: kid'

In [9]:
print(decode_review([1, 14, 20, 21]))

<START> this movie but


In [10]:
print(decode_review(X_train[0]))
print(y_train[0])
print(decode_review(X_train[1]))
print(y_train[1])

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for retail and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also congratulations to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the praising list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be p

## Pad (or Truncate) Sequences

In [14]:
sequence = [[1, 14, 20], [2, 3, 4, 5, 6]]
pad_sequences(sequence, maxlen=4, padding='pre', truncating='post')

array([[ 0,  1, 14, 20],
       [ 2,  3,  4,  5]])

In [ ]:
# กำหนดจำนวนคำที่ต้องการใช้ใน model จาก sequences หรือ ประโยคที่มีอยู่ใน dataset
X_train = pad_sequences(X_train, maxlen=maxlen)
X_test = pad_sequences(X_test, maxlen=maxlen)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (25000, 80)
X_test shape: (25000, 80)


In [24]:
print(X_train[0, :])
print(decode_review(X_train[0]))

[   15   256     4     2     7  3766     5   723    36    71    43   530
   476    26   400   317    46     7     4 12118  1029    13   104    88
     4   381    15   297    98    32  2071    56    26   141     6   194
  7486    18     4   226    22    21   134   476    26   480     5   144
    30  5535    18    51    36    28   224    92    25   104     4   226
    65    16    38  1334    88    12    16   283     5    16  4472   113
   103    32    15    16  5345    19   178    32]
that played the <UNK> of norman and paul they were just brilliant children are often left out of the praising list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you think the whole story was so lovely because it was true and was someone's life after all that was shared with us all


## Build Model

In [25]:
rnn_hidden_dim = 5
word_embedding_dim = 100
model_rnn = Sequential()
model_rnn.add(Embedding(max_features, word_embedding_dim))

model_rnn.add(SimpleRNN(rnn_hidden_dim,
                        kernel_initializer=initializers.RandomNormal(stddev=0.001),
                        recurrent_initializer=initializers.Identity(gain=1.0),
                        activation='tanh',
                        input_shape=X_train.shape[1:]))
# ปกติแล้ว input_shape = (25000, 80) ดังนั้นแล้ว shape ที่เข้ามาคือ (80,) เพื่อให้โมเดลได้เรียนรู้ข้อมูลอย่างมีประสิทธิภาพ
model_rnn.add(Dense(1, activation='sigmoid'))

In [26]:
model_rnn.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 100)         2000000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 5)                 530       
                                                                 
 dense (Dense)               (None, 1)                 6         
                                                                 
Total params: 2,000,536
Trainable params: 2,000,536
Non-trainable params: 0
_________________________________________________________________


In [27]:
rmsprop = keras.optimizers.RMSprop(learning_rate=0.0001)

model_rnn.compile(loss='binary_crossentropy',
                    optimizer=rmsprop,
                    metrics=['accuracy'])

In [28]:
model_rnn.fit(X_train, y_train,
              batch_size=batch_size,
              epochs=10,
              validation_data=(X_test, y_test))

Epoch 1/10
782/782 [==============================] - 62s 77ms/step - loss: 0.6091 - accuracy: 0.6829 - val_loss: 0.5723 - val_accuracy: 0.7285
Epoch 2/10
782/782 [==============================] - 63s 80ms/step - loss: 0.5123 - accuracy: 0.7831 - val_loss: 0.4999 - val_accuracy: 0.7754
Epoch 3/10
782/782 [==============================] - 65s 83ms/step - loss: 0.4317 - accuracy: 0.8286 - val_loss: 0.4571 - val_accuracy: 0.7959
Epoch 4/10
782/782 [==============================] - 66s 85ms/step - loss: 0.3810 - accuracy: 0.8528 - val_loss: 0.4221 - val_accuracy: 0.8165
Epoch 5/10
782/782 [==============================] - 63s 81ms/step - loss: 0.3401 - accuracy: 0.8728 - val_loss: 0.4092 - val_accuracy: 0.8169
Epoch 6/10
782/782 [==============================] - 62s 80ms/step - loss: 0.3080 - accuracy: 0.8852 - val_loss: 0.3994 - val_accuracy: 0.8231
Epoch 7/10
782/782 [==============================] - 65s 83ms/step - loss: 0.2838 - accuracy: 0.8940 - val_loss: 0.3700 - val_accuracy:

In [29]:
score, acc = model_rnn.evaluate(X_test, y_test, batch_size=batch_size)
print('Test lost score:', score)
print('Test accuracy score:', acc)

782/782 [==============================] - 9s 12ms/step - loss: 0.3787 - accuracy: 0.8398
Test lost score: 0.37874945998191833
Test accuracy score: 0.8398399949073792


## Predict

In [16]:
# เนื่องจาก Tokenizer เป็นคลาส เราเลยต้องสร้าง instance ขึ้นมาก่อน
new_text = ["I have recently seen the latest Spielberg movie, and it measures up to greats like E.T., Jaws, and Jurassic Park. Ready Player One is a movie about a virtual world called the OASIS whose creator, James Halliday (played by Mark Rylance) has died. In his will he creates a quest for control of the OASIS, which Parzival (played by Tye Sheridan) had devoted his life to winning. The quest is entirely 80s themed, and is absolutely great. All of the actors did a great job playing their roles. One of the best was Mark Rylance. He did an exceptional job as James Halliday, and the somewhat crazy man he was. Simon Pegg, who played Ogden Morrow, did a great job being annoyed at the weird kid always wasting his time. I loved all the 80s references in the movie. The music was amazing, as well as all of the cameos. Although I was upset at how much it changed from the book, it was still just as good, if not better. The best scene was the final battle, by far. I think that the moral is overused, yet important. It teaches that reality is more important than a game. Although it is necessary, it doesn't need to be in every movie and book a child ever reads. But, unlike most movies and books, it is subtle with this moral, and it does a good job of saying it's message without full out calling video games evil, like most movies incorrectly say. You will like this book if you were a teenager in the 80s, or even a teenager nowadays. It has a lot of 80s references, but almost every major reference can still be understood today because their about such well known things. All of the music is some of the most well-known 80s music ever, and all of the videogames are immortalized legends. It takes very little 80s or video game knowledge to understand this movie, but the more you have, the better. This movie is also rated PG-13, and I agree with that rating."]

tokenizer = Tokenizer(num_words=max_features, filters='!"#$%&()*+,./:;<=>?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(new_text)
new_sequences = tokenizer.texts_to_sequences(new_text)

# pad (or truncate) sequences to the same length
new_sequences_pad = pad_sequences(new_sequences, maxlen=maxlen)


In [18]:
test_text = ["I love this movie."]

In [21]:
print(new_sequences)

[[10, 30, 63, 64, 1, 65, 66, 7, 4, 3, 67, 68, 12, 69, 17, 70, 71, 72, 4, 73, 74, 75, 76, 31, 5, 2, 7, 32, 2, 77, 78, 79, 1, 33, 80, 81, 34, 35, 18, 19, 36, 37, 38, 82, 13, 20, 39, 21, 83, 2, 40, 84, 85, 6, 1, 33, 86, 87, 18, 19, 88, 89, 90, 91, 20, 92, 12, 93, 1, 40, 5, 94, 8, 95, 4, 5, 96, 22, 11, 6, 1, 97, 23, 2, 22, 14, 98, 41, 99, 31, 6, 1, 42, 9, 36, 37, 21, 23, 100, 101, 14, 15, 34, 35, 4, 1, 102, 103, 104, 21, 9, 105, 106, 107, 18, 108, 109, 23, 2, 22, 14, 110, 111, 43, 1, 112, 113, 114, 115, 20, 116, 10, 117, 11, 1, 8, 44, 13, 1, 7, 1, 24, 9, 118, 15, 45, 15, 11, 6, 1, 119, 46, 10, 9, 120, 43, 121, 122, 3, 123, 124, 1, 25, 3, 9, 47, 125, 15, 48, 49, 126, 50, 1, 42, 127, 9, 1, 128, 129, 19, 130, 10, 131, 26, 1, 51, 5, 132, 133, 52, 3, 134, 26, 135, 5, 53, 52, 136, 2, 54, 46, 3, 5, 137, 3, 138, 139, 12, 55, 13, 56, 7, 4, 25, 2, 140, 57, 141, 27, 142, 28, 58, 4, 143, 3, 5, 144, 59, 16, 51, 4, 3, 145, 2, 48, 14, 6, 146, 147, 148, 149, 150, 151, 152, 60, 153, 154, 17, 28, 58, 155, 1

In [25]:
decode_review([10, 30, 63])

'br be which'

In [26]:
decode_review(new_sequences[0])

"br be which only <START> story really of the <UNUSED> see their it had as can were me the well than we much one and <UNK> of all <UNK> been bad get <START> at will do by an for with they who so also i movie from but into <UNK> like people other a <START> at first great for with because how him most movie don't it made <START> like and its to then the and way film in a <START> make on <UNK> film this them her too one a <START> or is they who but on could any this that by an the <START> movies after think but is characters watch two for films character on <UNK> film this seen many just <START> being life plot never movie acting br little in <START> to about i <START> of <START> not is best that it's that in a <START> love out br is over just where did <UNUSED> show know <START> you <UNUSED> is has off that if some ever there <START> or does is <START> better your with end br still are <START> what and man here good <UNUSED> these are say and more good scene <UNK> when out <UNUSED> and w

In [67]:
print(new_sequences_pad)

[[  2  61 159   3  38   2 160   6   8  44  27 161  56 162 163 164  47  55
  165 166 167  41  32 168  45 169 170  11   6   1  24   5 171   6   1  28
  172   8  24  57   4  11   6   1 173 174 175 176   3 177 178 179   8  62
   60  54 180  12 181  16   7  27   1  53  29  30   1  50  16   7   5 182
  183 184   4  10 185  59  26 186]]


In [68]:
# predict
predict = model_rnn.predict(new_sequences_pad)
print(predict)

1/1 [==============================] - 0s 27ms/step
[[0.16144933]]
